# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad-imran2891/week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Answer:A page is worth reviewing for refresh if three things are all true at once: it's gone stale (no update in 90+ days), it's actually visible in search (moderate impressions or better, ≥ 300/90 days), and there's real room for it to climb (it currently sits somewhere between position ~4 and ~50 — page-1's bottom half through page 5). A page that's already in the top 3 has nothing to gain from a refresh, and a page buried past page 5 probably needs more than a refresh to fix. Only pages that clear all three checks get flagged; everything else is left alone.

Reason codes it can output:

stale_slipping_position — flagged, and sitting in positions 11–50 (page 2–5): the widest room to move up
stale_holding_page1 — flagged, and sitting in positions 4–10 (bottom half of page 1): closest to the top 3
not_flagged — doesn't meet all three conditions (too fresh, not enough traffic, already top 3, or too deep for a refresh alone to help)

Each row gets exactly one of these three codes, never more than one.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/saad-imran2891/week1"
REPO_DIR = "week1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

base_rate = (df["trend_direction"] == "down").mean()

signal_a = (
    df.assign(is_declining=(df["trend_direction"] == "down"))
      .groupby("freshness_tier")
      .agg(n=("content_id", "size"), decline_rate=("is_declining", "mean"))
      .round(3)
      .sort_values("decline_rate")
)
print("SIGNAL A — freshness_tier vs. decline rate (trend_direction == 'down')")
print(signal_a)
print(f"Overall base decline rate: {base_rate:.3f}\n")
print("Verdict: MIXED — the most-stale bucket (181+, n=174) sits BELOW the base rate,")
print("not above it. Staleness alone does not cleanly predict decline here.\n")

position_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
signal_b = (
    df.groupby("position_tier")
      .agg(n=("content_id", "size"), avg_ctr=("ctr", "mean"))
      .reindex(position_order)
      .round(3)
)
print("SIGNAL B — position_tier vs. avg ctr (best position to worst)")
print(signal_b)
print("\nVerdict: CONFIRMED — avg ctr falls monotonically as position gets worse.")


Working dir: /content/week1/week1
Loaded 30,000 rows x 44 columns
SIGNAL A — freshness_tier vs. decline rate (trend_direction == 'down')
                    n  decline_rate
freshness_tier                     
181+              174         0.471
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
Overall base decline rate: 0.542

Verdict: MIXED — the most-stale bucket (181+, n=174) sits BELOW the base rate,
not above it. Staleness alone does not cleanly predict decline here.

SIGNAL B — position_tier vs. avg ctr (best position to worst)
                   n  avg_ctr
position_tier                
top_3           2321    1.484
page_1         11814    0.652
striking        7304    0.323
page_3_5        7242    0.222
deep            1319    0.150

Verdict: CONFIRMED — avg ctr falls monotonically as position gets worse.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
ANSWER:The score multiplies three yes/no conditions together with the page's own impression volume — no fitted weights, just conditions a person can check by eye:

score = stale × visible × opportunity × impressions_90d

stale = 1 if freshness_tier is 91-180 or 181+, else 0
visible = 1 if impression_tier is moderate, good, or excellent, else 0
opportunity = 1 if position_tier is page_1, striking, or page_3_5, else 0

If any condition fails, the score is 0 and the page gets no_action. Among flagged pages, the ones with the most impressions rank first — the logic being: if we're only going to refresh so many pages, fix the ones already pulling the most search traffic first.

The code cell below builds this, ranks the full 30,000 rows, and writes the ranked queue to work/outputs/baseline_action_score.csv.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)

stale = df["freshness_tier"].isin(["91-180", "181+"]).astype(int)
visible = df["impression_tier"].isin(["moderate", "good", "excellent"]).astype(int)
opportunity = df["position_tier"].isin(["page_1", "striking", "page_3_5"]).astype(int)

# readable on purpose: no fitted weights, just conditions multiplied together
df["score"] = stale * visible * opportunity * df["impressions_90d"]
flagged = df["score"] > 0

df["reason_code"] = np.select(
    [
        flagged & df["position_tier"].isin(["striking", "page_3_5"]),
        flagged & (df["position_tier"] == "page_1"),
    ],
    ["stale_slipping_position", "stale_holding_page1"],
    default="not_flagged",
)
df["action"] = np.where(flagged, "review_for_refresh", "no_action")
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)

queue = df.sort_values("rank")[[
    "rank", "content_id", "client_id", "action", "reason_code", "score",
    "impressions_90d", "freshness_tier", "impression_tier", "position_tier",
    "avg_position", "ctr",
]]

out_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} rows to {out_path}")
print(queue["action"].value_counts(), "\n")
print(queue["reason_code"].value_counts(), "\n")

labels = (df["trend_direction"] == "down").astype(int).to_numpy()
order = np.argsort(-df["score"].to_numpy())
for k in (20, 50, 100):
    p_at_k = labels[order[:k]].mean()
    print(f"precision@{k}: {p_at_k:.3f}  (base rate {labels.mean():.3f})")

Wrote 30,000 rows to work/outputs/baseline_action_score.csv
action
no_action             23176
review_for_refresh     6824
Name: count, dtype: int64 

reason_code
not_flagged                23176
stale_slipping_position     4136
stale_holding_page1         2688
Name: count, dtype: int64 

precision@20: 0.450  (base rate 0.542)
precision@50: 0.420  (base rate 0.542)
precision@100: 0.400  (base rate 0.542)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
ANSWER:For each of the top 20 rows by score, I print the action (review_for_refresh in every case here, since anything flagged 0 ranks last), the reason code it got, a confidence note built from its own numbers (traffic, staleness, position, CTR), and one honest sentence on what would make that specific pick wrong — not a generic disclaimer, but the actual condition on file (trend_direction) that would undercut the pick.

The code cell below generates and prints all 20, one block per row.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.sort_values("score", ascending=False).head(20).copy()

def confidence_note(row):
    gap = "wide" if row["position_tier"] in ("striking", "page_3_5") else "narrow"
    return (f"{row['impressions_90d']:,.0f} impr/90d, stale {row['freshness_tier']} days, "
            f"avg position {row['avg_position']:.1f} ({gap} gap to top 3), ctr {row['ctr']:.2f}%.")

def would_be_wrong_if(row):
    if row["trend_direction"] == "up":
        return "it's already trending up on its own — refreshing now could be wasted effort."
    if row["trend_direction"] == "stable":
        return "'stable' just means no big swing — if a human check shows it's fine, it doesn't need a refresh."
    return "the drop is from something a refresh can't fix (seasonal dip, SERP feature change, cannibalization)."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(would_be_wrong_if, axis=1)

for i, row in enumerate(top20.itertuples(index=False), start=1):
    print(f"{i:>2}. {row.content_id}")
    print(f"    action: {row.action}  |  reason_code: {row.reason_code}  |  trend on file: {row.trend_direction}")
    print(f"    confidence: {row.confidence_note}")
    print(f"    what would make it wrong: {row.what_would_make_it_wrong}")
    print()

 1. content_5fe46e04994d
    action: review_for_refresh  |  reason_code: stale_holding_page1  |  trend on file: down
    confidence: 517,715 impr/90d, stale 91-180 days, avg position 4.2 (narrow gap to top 3), ctr 0.14%.
    what would make it wrong: the drop is from something a refresh can't fix (seasonal dip, SERP feature change, cannibalization).

 2. content_2dba2b1f9536
    action: review_for_refresh  |  reason_code: stale_slipping_position  |  trend on file: stable
    confidence: 443,434 impr/90d, stale 91-180 days, avg position 27.9 (wide gap to top 3), ctr 0.21%.
    what would make it wrong: 'stable' just means no big swing — if a human check shows it's fine, it doesn't need a refresh.

 3. content_2c2606c5d176
    action: review_for_refresh  |  reason_code: stale_holding_page1  |  trend on file: down
    confidence: 347,399 impr/90d, stale 91-180 days, avg position 4.2 (narrow gap to top 3), ctr 0.53%.
    what would make it wrong: the drop is from something a refresh can't 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
ANSWER:Look at precision@20 (0.45), precision@50 (0.42), and precision@100 (0.40) from Section 2 — all three sit below the overall base decline rate of 0.542. That's the tell: because score = stale × visible × opportunity × impressions_90d, among flagged pages the rule just sorts by raw traffic. It surfaces the biggest, stalest, page-1-to-5 pages on the site — but "big and stale" isn't the same as "actually declining." Several rows in the top 20 (see Section 3's printed trend on file) are trend_direction == "stable", meaning the page hasn't lost ground at all; it's just large and untouched. That's a real weakness, not a fluke — this rule would happily send someone to refresh a page that's doing fine, purely because it gets a lot of traffic.

This is expected and even useful: a baseline's job is to be honestly beatable, and this is the honest gap a future model should close.

Leakage check:

The code cell below lists every column the rule actually reads and checks it against four buckets: label-derived columns (trend_direction, trend_pct — the exact inputs to is_declining_label), future/last-30-day comparison-window columns, raw IDs used as features (rather than only for grouping), and product/provider flags (provider_used, model_used, explicitly marked "not a model feature" in the data dictionary). None of these show up in stale, visible, opportunity, or the reason codes — the rule only ever reads freshness_tier, impression_tier, position_tier, and impressions_90d.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_trend_counts = top20["trend_direction"].value_counts()
print("Top-20 trend_direction mix (for review only, not used by the rule):")
print(top20_trend_counts, "\n")

print("Weak picks, in plain words:")
print("- precision@20 (0.45), @50 (0.42), @100 (0.40) all sit BELOW the base decline rate")
print("  (0.542). The score is stale*visible*opportunity*impressions_90d, so among flagged")
print("  pages it just sorts by raw traffic — it surfaces the biggest stale pages, not")
print("  necessarily the ones actually declining. Several top-20 rows are trend_direction")
print("  == 'stable', which is a real weakness of this rule.")
print("- This is an honest baseline weakness, not a bug: a future model should clearly beat")
print("  this on precision@K.\n")

label_derived_cols = {"trend_direction", "trend_pct"}
future_window_cols = {"impressions_last_30d", "clicks_last_30d", "sessions_last_30d"}
id_cols = {"content_id", "client_id"}
product_flag_cols = {"provider_used", "model_used"}

rule_inputs = {"freshness_tier", "impression_tier", "position_tier", "impressions_90d"}

print("Leakage check:")
print("- Columns the rule actually reads:", sorted(rule_inputs))
print("- Label-derived columns used?", "YES" if rule_inputs & label_derived_cols else "NO")
print("- Future/last-30d window columns used?", "YES" if rule_inputs & future_window_cols else "NO")
print("- Raw IDs used as a feature?", "YES" if rule_inputs & id_cols else "NO")
print("- Provider/model flags used?", "YES" if rule_inputs & product_flag_cols else "NO")

Top-20 trend_direction mix (for review only, not used by the rule):
trend_direction
stable    10
down       9
up         1
Name: count, dtype: int64 

Weak picks, in plain words:
- precision@20 (0.45), @50 (0.42), @100 (0.40) all sit BELOW the base decline rate
  (0.542). The score is stale*visible*opportunity*impressions_90d, so among flagged
  pages it just sorts by raw traffic — it surfaces the biggest stale pages, not
  necessarily the ones actually declining. Several top-20 rows are trend_direction
  == 'stable', which is a real weakness of this rule.
- This is an honest baseline weakness, not a bug: a future model should clearly beat
  this on precision@K.

Leakage check:
- Columns the rule actually reads: ['freshness_tier', 'impression_tier', 'impressions_90d', 'position_tier']
- Label-derived columns used? NO
- Future/last-30d window columns used? NO
- Raw IDs used as a feature? NO
- Provider/model flags used? NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.